In [1]:
import numpy as np 
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import xgboost
import optuna

from sklearn.model_selection import train_test_split,StratifiedKFold
from sklearn.metrics import accuracy_score,classification_report,roc_auc_score,roc_curve,f1_score
from xgboost import XGBClassifier,callback

In [2]:
train = pd.read_csv('/kaggle/input/competitions/playground-series-s6e3/train.csv')
test = pd.read_csv('/kaggle/input/competitions/playground-series-s6e3/test.csv')

In [3]:
train.head()

,id,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,0,Male,0,Yes,Yes,29,Yes,No,DSL,Yes,...,Yes,Yes,No,No,One year,Yes,Mailed check,60.10,1653.85,No
1,1,Male,0,Yes,Yes,58,Yes,No,DSL,Yes,...,No,Yes,Yes,No,Two year,No,Credit card (automatic),69.50,3778.20,No
2,2,Male,0,Yes,No,58,Yes,Yes,Fiber optic,No,...,No,No,Yes,Yes,Month-to-month,Yes,Electronic check,100.40,5841.35,No
3,3,Female,0,No,No,1,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,69.70,70.70,Yes
4,4,Female,0,No,No,1,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.45,70.45,Yes


In [4]:
test_id = test['id']

In [5]:
train.head()

,id,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,0,Male,0,Yes,Yes,29,Yes,No,DSL,Yes,...,Yes,Yes,No,No,One year,Yes,Mailed check,60.10,1653.85,No
1,1,Male,0,Yes,Yes,58,Yes,No,DSL,Yes,...,No,Yes,Yes,No,Two year,No,Credit card (automatic),69.50,3778.20,No
2,2,Male,0,Yes,No,58,Yes,Yes,Fiber optic,No,...,No,No,Yes,Yes,Month-to-month,Yes,Electronic check,100.40,5841.35,No
3,3,Female,0,No,No,1,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,69.70,70.70,Yes
4,4,Female,0,No,No,1,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.45,70.45,Yes


In [6]:
train_cols = train.select_dtypes(['object']).columns
test_cols = test.select_dtypes(['object']).columns

In [7]:
train = pd.get_dummies(data=train,columns=train_cols[:-1],drop_first=True,dtype=int)
test = pd.get_dummies(data=test,columns=test_cols,drop_first=True,dtype=int)

In [8]:
train.drop(columns=['id'],inplace = True)
test.drop(columns=['id'],inplace = True)

In [9]:
train['Churn'] = train['Churn'].map({'No':0,'Yes':1})

In [10]:
X = train.drop(columns=['Churn'])
y = train['Churn']

In [11]:
# model = XGBClassifier(
#     learning_rate=0.02,
#     tree_method='hist',
#     random_state=42,
#     subsample=0.8,
#     colsample_bytree=0.4,
#     n_estimators=3300,
#     max_depth=5,
#     gamma=0.3,
#     min_child_weight=6,
#     device='cuda',
#     early_stopping_rounds=150,
#     reg_alpha=3,
#     enable_categorical=True
# )

In [12]:
# oof_preds = np.zeros(len(X)) 
# # test_preds_total: To store the average predictions for the test set (for submission)
# test_preds_total = np.zeros(len(test))
# # Initialize the S-KFold strategy
# n_s = 12
# skf = StratifiedKFold(n_splits=n_s, shuffle=True, random_state=42)
# for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
#     X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
#     y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]
    
#     model.fit(
#         X_train, y_train,
#         eval_set=[(X_val, y_val)],
#         verbose=0
#     )
    
#     pre_y_xgb = model.predict_proba(X_val)[:, 1]
#     print(f'Score of Fold {fold} XGB: {roc_auc_score(y_val,pre_y_xgb):.5f} %')

    

#     test_preds_total += model.predict_proba(test)[:, 1]/n_s
#     oof_preds[val_idx] = pre_y_xgb
#     print(f"Fold {fold} finished.")

In [13]:
# print(f'XGB Train score:{roc_auc_score(y,oof_preds):.5f}')

In [14]:
# preditions = test_preds_total

In [15]:
# df = pd.DataFrame({
#     'id':test_id,
#     'Churn':preditions
# })

In [16]:
# df.head()

In [17]:
# df.to_csv('sub_3.csv',index=False)

In [18]:
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size = 0.2,random_state=42,stratify=y)

In [19]:
def objective(trial):
  params = {
      "objective":"binary:logistic",
      "eval_metric":"auc",
      "tree_method":"hist",
      "random_state":42,
      "verbosity":0,
      "device":"cuda",
      "max_depth":trial.suggest_int("max_depth",3,12,step=1),
      "learning_rate":trial.suggest_float("learning_rate",1e-3,0.1,log=True),
      "min_child_weight":trial.suggest_int("min_child_weight",1,20),
      "gamma":trial.suggest_float("gamma",0,5),
      "subsample":trial.suggest_float("subsample",0.5,1.0),
      "colsample_bytree":trial.suggest_float("colsample_bytree",0.5,1.0),
      "reg_alpha": trial.suggest_float("reg_alpha", 1e-8, 10.0, log=True),
      "reg_lambda": trial.suggest_float("reg_lambda", 1e-8, 10.0, log=True),
      "scale_pos_weight": trial.suggest_float("scale_pos_weight", 0.5, 10.0)
  }

  kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
  scores = []
  best_iters = []
  for fold, (train_idx, valid_idx) in enumerate(kf.split(X_train,y_train)):
        X_tr, X_val = X_train.iloc[train_idx], X_train.iloc[valid_idx]
        y_tr, y_val = y_train.iloc[train_idx], y_train.iloc[valid_idx]

        model = XGBClassifier(
            **params,
            n_estimators=5000,
            callbacks=[
                callback.EarlyStopping(
                    rounds=100,
                    save_best=True
                )
            ]
        )
        model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=False)

        proba = model.predict_proba(X_val)[:, 1]
        score = roc_auc_score(y_val, proba)
        scores.append(score)

        best_iters.append(model.best_iteration + 1 if model.best_iteration is not None else 5000)
        trial.report(np.mean(scores), step=fold)
        if trial.should_prune():
            raise optuna.exceptions.TrialPruned()
  trial.set_user_attr("avg_best_iteration", float(np.mean(best_iters)))
  return float(np.mean(scores))


In [20]:
study = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=42),
    pruner=optuna.pruners.MedianPruner(n_warmup_steps=5)
)
study.optimize(objective,n_trials=50)

[I 2026-03-24 08:10:23,425] A new study created in memory with name: no-name-535ec218-5796-4a81-8d6f-5740a312a087
[I 2026-03-24 08:10:38,079] Trial 0 finished with value: 0.9156294896969323 and parameters: {'max_depth': 6, 'learning_rate': 0.07969454818643935, 'min_child_weight': 15, 'gamma': 2.993292420985183, 'subsample': 0.5780093202212182, 'colsample_bytree': 0.5779972601681014, 'reg_alpha': 3.3323645788192616e-08, 'reg_lambda': 0.6245760287469893, 'scale_pos_weight': 6.210592611560483}. Best is trial 0 with value: 0.9156294896969323.
[I 2026-03-24 08:11:38,112] Trial 1 finished with value: 0.9130611540210897 and parameters: {'max_depth': 10, 'learning_rate': 0.0010994335574766201, 'min_child_weight': 20, 'gamma': 4.162213204002109, 'subsample': 0.6061695553391381, 'colsample_bytree': 0.5909124836035503, 'reg_alpha': 4.4734294104626844e-07, 'reg_lambda': 5.472429642032198e-06, 'scale_pos_weight': 5.48518610050626}. Best is trial 0 with value: 0.9156294896969323.
[I 2026-03-24 08:13

In [21]:
best_params = study.best_trial.params
best_score = study.best_trial.value
print(best_params, best_score)

{'max_depth': 4, 'learning_rate': 0.0464162536906739, 'min_child_weight': 17, 'gamma': 1.6308332526987914, 'subsample': 0.647431867444064, 'colsample_bytree': 0.8769545229170373, 'reg_alpha': 4.3548661831192793e-07, 'reg_lambda': 0.011377817984195545, 'scale_pos_weight': 0.5447473561688188} 0.9162766493204086


In [22]:
best_iter = int(study.best_trial.user_attrs["avg_best_iteration"])

In [23]:
final_model = XGBClassifier(
    **study.best_trial.params,
    objective="binary:logistic",
    eval_metric="auc",
    tree_method="hist",
    random_state=42,
    verbosity=0,
    device="cuda",
    n_estimators=best_iter
)

final_model.fit(X_train, y_train)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.8769545229170373, device='cuda',
              early_stopping_rounds=None, enable_categorical=False,
              eval_metric='auc', feature_types=None, feature_weights=None,
              gamma=1.6308332526987914, grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.0464162536906739,
              max_bin=None, max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=4, max_leaves=None,
              min_child_weight=17, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=2421, n_jobs=None,
              num_parallel_tree=None, ...)

In [24]:
test_proba = final_model.predict_proba(test)[:, 1]

In [25]:
submission = pd.DataFrame({
    "id": test_id,
    "Churn": test_proba
})
submission.to_csv("submission.csv", index=False)